# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 4: Prompt Engineering</font>

# <font color="#003660">Notebook 3: Prompting with the OpenAI-Compatible API</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... know why the OpenAI-compatible API is a de facto standard and how to switch between model providers, <br>
        ... can send messages, system prompts, and few-shot examples with <code>chat.completions</code>, <br>
        ... can control the generation with parameters such as <code>temperature</code>, <code>seed</code>, <code>max_tokens</code>, and <code>stop</code>, <br>
        ... can switch the thinking of reasoning models on and off with <code>reasoning_effort</code>.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [Raschka (2024): Building a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch)
* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Huggingface (2024): Open-Source AI Cookbook](https://huggingface.co/learn/cookbook/index)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

These documentations can help you for your projects:
* Decoding strategies: [Hugging Face (2025): Generation strategies](https://huggingface.co/docs/transformers/generation_strategies)
* Chat templates: [Hugging Face: Chat templates](https://huggingface.co/docs/transformers/chat_templating)
* Ollama API: [Ollama API documentation](https://docs.ollama.com/api)
* OpenAI API: [OpenAI Python SDK](https://platform.openai.com/docs/api-reference/chat/create?lang=python)
* Ollama provides OpenAI API compatible endpoints: [Ollama](https://github.com/ollama/ollama), [Ollama Python](https://github.com/ollama/ollama-python), [Ollama OpenAI Compatibility](https://docs.ollama.com/api/openai-compatibility)
* HuggingFace provides OpenAI API compatible endpoints: [HuggingFace Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), [HuggingFace Endpoints](https://huggingface.co/docs/inference-endpoints/index)

## Where We Are

In 4.01 and 4.02 you wrote prompt templates by hand and used Ollama's own API. In your projects, you will mostly use another API: the **OpenAI-compatible API**. This notebook brings no new prompting concepts. We do what you already know with this API, and we check that it builds the same templates behind the scenes.

## Setup

The same setup as in 4.01 and 4.02. If you work in a new Colab runtime, run all cells of this section. On macOS or Windows with the Ollama app, skip Steps 1 and 2.

### Step 1: Install Ollama

In [1]:
!sudo apt-get install -y pciutils zstd

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

### Step 2: Start the Ollama server

In [3]:
import subprocess
import time

# the server writes its log into a file; a pipe that nobody reads could fill up and block the server
ollama_log = open("ollama.log", "w")
process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)

# give the server a few seconds to start
time.sleep(5)

### Step 3: Pull the chat model

We use the chat model from 4.02. A reasoning model follows in Part 4.

In [4]:
# You can work on a small gpu (e.g., T4) --> download will take a while (1 minute)
!ollama pull llama3.2:1b

In [5]:
CHAT_MODEL = "llama3.2:1b"  # chat model: Llama 3.2 1B after additional training on conversations

### Step 4: Connect from Python, with a magic trick

Here is the magic trick: Ollama, like most model providers and open-source projects, supports the **OpenAI-compatible API**. OpenAI designed this interface for its own models, and it has become a de facto standard. Ollama, [vLLM](https://docs.vllm.ai/), [Hugging Face Inference Providers](https://huggingface.co/docs/inference-providers/index?python-clients=openai&javascript-clients=openai), and many cloud providers accept the same requests. You write your code once with the `openai` Python package, and you switch between a local model and a cloud model by changing two lines: the address of the server (`base_url`) and the key (`api_key`).

Using the OpenAI API format does not mean using OpenAI's models. Everything in this notebook runs on open models.

We install the `openai` package and point the client to our local Ollama server instead of OpenAI's servers.

In [6]:
!pip install openai

In [7]:
from openai import OpenAI

# address of our local Ollama server
OLLAMA_HOST = "http://127.0.0.1:11434"

client = OpenAI(
    base_url=f"{OLLAMA_HOST}/v1",  # our local server instead of OpenAI's servers
    api_key="ollama",              # Ollama does not check the key, but the package needs one
)

Let's check the connection: the server lists the models it provides.

In [8]:
for model in client.models.list():
    print(model.id)

### Two small helpers

`what_the_model_sees()` is the helper from 4.02: it asks Ollama for the text that is built from a list of messages, without generating anything.

`score()` works like in 4.01 and 4.02, but with messages instead of a raw template: it fills the placeholder `{text}` in every message with each example, sends the messages with `client.chat.completions.create`, and counts the correct answers.

In [9]:
import re
import requests

GREEN, ORANGE, GREY, RESET = "\033[92m", "\033[33m", "\033[90m", "\033[0m"
SPECIAL_TOKENS = re.compile(r"(<\|[^|>]+\|>|</?think>|<start_of_turn>|<end_of_turn>)")


def highlight(text, color="", newlines=False):
    """Highlights special tokens (inverse video) and, if wanted, shows line breaks as ↵."""
    if newlines:
        text = text.replace("\n", "↵\n")
    text = SPECIAL_TOKENS.sub(lambda m: f"\033[7m{m.group(0)}\033[27m", text)
    return f"{color}{text}{RESET}"


def what_the_model_sees(model, messages, think=None, newlines=True):
    """Lets Ollama build the prompt from a list of messages, without generating anything."""
    payload = {"model": model, "messages": messages, "stream": False, "_debug_render_only": True}
    if think is not None:
        payload["think"] = think
    response = requests.post(f"{OLLAMA_HOST}/api/chat", json=payload, timeout=300)
    response.raise_for_status()
    text = response.json()["_debug_info"]["rendered_template"]
    print(highlight(text, newlines=newlines))


def check(answer, expected):
    """Looks for the expected answer behind "The answer is" or, if the model did not write that, in the first line."""
    answer = answer.split("</think>")[-1]  # ignore the reasoning of reasoning models
    position = answer.lower().find("answer is")
    if position >= 0:
        answer = answer[position + len("answer is"):]
    first_line = next((line.strip() for line in answer.splitlines() if line.strip()), "")
    correct = re.search(rf"\b{re.escape(expected)}\b", first_line, re.IGNORECASE) is not None
    return correct, first_line


def fill(messages, text):
    """Replaces the placeholder {text} in every message."""
    return [{**message, "content": message["content"].replace("{text}", text)} for message in messages]


def score(messages, examples, model=CHAT_MODEL, temperature=0, seed=42, max_tokens=20, **params):
    """Sends the messages for every example and counts the correct answers."""
    correct, tokens = 0, 0
    for text, expected in examples:
        response = client.chat.completions.create(
            model=model, messages=fill(messages, text), temperature=temperature, seed=seed, max_tokens=max_tokens, **params
        )
        ok, answer = check(response.choices[0].message.content or "", expected)
        correct += ok
        tokens += response.usage.completion_tokens
        mark = f"{GREEN}✓{RESET}" if ok else f"{ORANGE}✗{RESET}"
        print(f"{mark} expected: {expected:<10} model: {answer[:70]}")
    print(f"\n{correct}/{len(examples)} correct, {tokens} tokens generated")

## Part 1: One Request

You know the messages format from 4.02. The OpenAI-compatible API takes exactly the same list of messages:

In [10]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0, seed=42)
print(response.choices[0].message.content)

The response is a JSON object. Besides the answer, it tells you why the generation stopped (`finish_reason`) and how many tokens were used (`usage`):

In [11]:
print(response.model_dump_json(indent=4))

And behind the scenes? Ollama applied the same chat template that you wrote by hand in 4.02:

In [12]:
what_the_model_sees(CHAT_MODEL, messages)

The OpenAI-compatible API is just another door to the same template.

## Part 2: System Prompts and Few-Shot Examples as Messages

A system prompt is the first message in the list. Few-shot examples are earlier `user` and `assistant` messages that you write yourself: the model sees them as earlier turns of the conversation. Let's do the sentiment task from 4.01 with the chat model:

In [13]:
# ten short movie reviews with their sentiment
REVIEWS = [
    ("A wonderful film with brilliant actors.", "positive"),
    ("The plot was boring and far too long.", "negative"),
    ("I laughed from start to finish.", "positive"),
    ("A complete waste of two hours.", "negative"),
    ("Not bad at all, I would watch it again.", "positive"),
    ("The ending made no sense, and the dialogue was awful.", "negative"),
    ("Beautiful music and a touching story.", "positive"),
    ("I fell asleep halfway through.", "negative"),
    ("It was not as good as everyone said.", "negative"),
    ("My kids loved it, and so did I.", "positive"),
]

In [14]:
zero_shot = [
    {"role": "system", "content": "Classify the sentiment of the movie review. Answer with positive or negative only."},
    {"role": "user", "content": "{text}"},
]
score(zero_shot, REVIEWS)

### Your turn

1. Add one or more examples as earlier `user` and `assistant` messages. Use your own reviews, not reviews from the list.
2. Compare the score with your best base-model template from 4.01. Does the chat model need the examples?
3. Look at what the model sees for your messages: where do your examples end up in the chat template?

In [15]:
# TODO: add your examples as earlier user and assistant messages
few_shot = [
    {"role": "system", "content": "Classify the sentiment of the movie review. Answer with positive or negative only."},
    {"role": "user", "content": "{text}"},
]
score(few_shot, REVIEWS)

In [16]:
what_the_model_sees(CHAT_MODEL, fill(few_shot, REVIEWS[0][0]))

## Part 3: Controlling the Generation

The decoding parameters from Session 3 are part of every request:

| Parameter | What it does |
|---|---|
| `temperature` | flattens (> 1) or sharpens (< 1) the next-token distribution; `0` means greedy decoding |
| `top_p` | samples only from the most likely tokens that together reach this probability |
| `seed` | fixes the random numbers, so that sampling gives the same result again |
| `max_tokens` | the maximum number of generated tokens |
| `stop` | a list of stop sequences, as in 4.01 |

The response tells you in `finish_reason` why the generation stopped: `length` (maximum number of tokens) or `stop` (end-of-turn token or stop sequence).

In [17]:
messages = [{"role": "user", "content": "Write a short poem about the sea."}]

for seed in [1, 2, 3]:
    response = client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=1.0, seed=seed, max_tokens=40)
    print(f"seed {seed}, finish_reason: {response.choices[0].finish_reason}")
    print(response.choices[0].message.content, end="\n\n")

### Your turn

**Predict** before you run it: what changes with `temperature=0`? What is the `finish_reason` with `max_tokens=200`, and what with `stop=["\n"]`?

In [18]:
# TODO: change temperature, max_tokens, and stop, and compare the outputs and the finish_reason

Not every Ollama option exists in the OpenAI-compatible API. For example, `num_ctx` (the context window) and `repeat_penalty` are missing. For the context window, you can set the environment variable `OLLAMA_CONTEXT_LENGTH` when you start the Ollama server.

## Part 4: Reasoning Models

Now the reasoning model from 4.02. It needs about 2.7 GB, and it fits next to the chat model on a Colab T4 GPU.

In [19]:
!ollama pull qwen3.5:2b

In [20]:
REASONING_MODEL = "qwen3.5:2b"

With the OpenAI-compatible API, you switch thinking on with `reasoning_effort`. Ollama returns the reasoning in an extra field called `reasoning`. We do not set a temperature, so Ollama uses the settings recommended for Qwen3.5.

In [ ]:
riddle = [{"role": "user", "content": "When I was 6 my sister was half my age. Now I'm 70. How old is my sister?"}]

response = client.chat.completions.create(model=REASONING_MODEL, messages=riddle, reasoning_effort="high")
message = response.choices[0].message
print(highlight(getattr(message, "reasoning", None) or "", ORANGE))
print("-" * 40)
print(message.content)
print(f"\n{GREY}[{response.usage.completion_tokens} tokens generated]{RESET}")

With `reasoning_effort="none"`, thinking is switched off. Behind the scenes, the template gets the empty thinking block from 4.02:

In [ ]:
response = client.chat.completions.create(model=REASONING_MODEL, messages=riddle, reasoning_effort="none")
print(response.choices[0].message.content)
print(f"\n{GREY}[{response.usage.completion_tokens} tokens generated]{RESET}")

### Your turn

Score your few-shot messages from Part 2 with the reasoning model and thinking switched off. Then try the zero-shot messages. Does the larger model still need the examples?

In [ ]:
# TODO: score your messages with the reasoning model
score(few_shot, REVIEWS, model=REASONING_MODEL, reasoning_effort="none")

## Part 5: Switching the Provider

This is the payoff of the magic trick. All the code in this notebook works with other providers if you change `base_url`, `api_key`, and the model name. Some examples (not to run now):

```python
import os

# a cloud provider, for example Hugging Face Inference Providers (needs a Hugging Face token)
client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])
response = client.chat.completions.create(model="openai/gpt-oss-20b", messages=messages)

# a vLLM server that you started yourself
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# OpenAI itself: the key is read from the environment variable OPENAI_API_KEY
client = OpenAI()
```

Never write an API key into a notebook. Use environment variables or, on Colab, the *Secrets* in the left sidebar.

Two differences remain between providers. First, the template: every model family has its own, and the provider applies it for you, so `what_the_model_sees()` only works with Ollama. Second, not every provider supports every parameter, for example `seed` or `reasoning_effort`. For base models, there is also the older **completions** endpoint (`client.completions.create`), which takes a text prompt instead of messages, but most providers today only offer chat models.

## Questions?

Now is the time for questions! Is anything about the OpenAI-compatible API, its parameters, or the connection to the templates unclear?

## What You Learned

* The **OpenAI-compatible API** is a de facto standard: the same code works with Ollama, vLLM, Hugging Face Inference Providers, OpenAI, and many others. You switch with `base_url`, `api_key`, and the model name.
* `client.chat.completions.create` takes the same **list of messages** as Ollama's chat API. The provider turns it into the model's **chat template**, the same text you wrote by hand in 4.02.
* System prompts and few-shot examples are just messages: the system message comes first, and examples are earlier `user` and `assistant` turns.
* `temperature`, `top_p`, `seed`, `max_tokens`, and `stop` control the generation, and `finish_reason` tells you why it stopped.
* `reasoning_effort` switches the thinking of reasoning models on and off, and the reasoning comes back in an extra field.